In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction==2026.11.11 --extra-index-url https://download.easydiffraction.org/pypi/simple/


# CeCoAl₃ on POLARIS

&nbsp; &nbsp; &nbsp; &nbsp; &nbsp; &nbsp; 

Fit CeCoAl₃ to time-of-flight neutron powder data from POLARIS at ISIS: structure and profile,
with a 24-term Chebyshev background.

The project is the catalogue case `pd-neut-tof_cecoal-polaris`, the one the command line and the
app's Examples open.

In [2]:
import easydiffraction as edi

## Load the project

In [3]:
project_path = edi.download_data('pd-neut-tof_cecoal-polaris', destination='data')
project = edi.Project.load(project_path)
structure = project.structures['cecoal']
experiment = project.experiments['polaris']

Data 'pd-neut-tof_cecoal-polaris' is at data/pd-neut-tof_cecoal-polaris-6a9fe6f6a3d7/pd-neut-tof_cecoal-polaris


The project marks what its fit frees. The fit below frees it in stages, so start with all of it
fixed.

In [4]:
for parameter in project.free_parameters:
    parameter.free = False

In [5]:
project.display.pattern('polaris')

View(html=<599,442 characters>, height=480)

## Perform the analysis

The fit goes in stages, freeing one group of parameters at a time.

### Scale, time-of-flight calibration and cell

The scale, the DIFC offset and its quadratic term, and the three cell lengths put the peaks in
place and at the right height.

In [6]:
experiment.linked_structures['cecoal'].scale.free = True
experiment.instrument.calib_d_to_tof_offset.free = True
experiment.instrument.calib_d_to_tof_quadratic.free = True
structure.cell.length_a.free = True
structure.cell.length_b.free = True
structure.cell.length_c.free = True

In [7]:
project.display.parameters.free()

#,datablock,category,entry,parameter,value,s.u.,min,max,units
1,cecoal,cell,,length_a,7.65,,0,30,Å
2,cecoal,cell,,length_b,4.05,,0,30,Å
3,cecoal,cell,,length_c,6.91,,0,30,Å
4,polaris,instrument,,calib_d_to_tof_offset,0,,-inf,inf,µs
5,polaris,instrument,,calib_d_to_tof_quadratic,-1.92194,,-inf,inf,µs/Å²
6,polaris,linked_structure,cecoal,scale,2.00×10⁴,,-inf,inf,


In [8]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_cecoal-polaris, 3692 pts fitted / 3693 loaded, 6 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       11.38        84.57
      1      0.785      4.13        11.13   86.8% ↓
  done · converged · 1 iters · reduced χ² 11.13 · Rwp 4.13 % · 0.89 s


View(html=<87,586 characters>, height=376)

In [9]:
project.display.pattern('polaris')

View(html=<597,616 characters>, height=480)

### Background

The first twelve polynomial terms (orders 0 to 11) follow the level between the peaks; the
higher ones stay at zero.

In [10]:
for term in list(experiment.background_terms)[:12]:
    term.coef.free = True

In [11]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_cecoal-polaris, 3692 pts fitted / 3693 loaded, 18 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        4.13        11.17
      1      0.818      2.31         3.51   68.5% ↓
  done · converged · 1 iters · reduced χ² 3.51 · Rwp 2.31 % · 0.92 s


View(html=<100,324 characters>, height=376)

In [12]:
project.display.pattern('polaris')

View(html=<617,129 characters>, height=480)

### Peak widths

In [13]:
experiment.peak.broad_gauss_sigma_1.free = True
experiment.peak.broad_lorentz_gamma_1.free = True

In [14]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_cecoal-polaris, 3692 pts fitted / 3693 loaded, 20 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        2.31         3.51
      1      0.889      2.31         3.51   0.2% ↓
  done · converged · 1 iters · reduced χ² 3.51 · Rwp 2.31 % · 1.00 s


View(html=<106,247 characters>, height=376)

In [15]:
project.display.pattern('polaris')

View(html=<617,238 characters>, height=480)

### Atom positions

The coordinates the symmetry leaves free.

In [16]:
for site in structure.atom_sites:
    for coordinate in (site.fract_x, site.fract_y, site.fract_z):
        coordinate.free = not coordinate.symmetry_constrained

In [17]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_cecoal-polaris, 3692 pts fitted / 3693 loaded, 25 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        2.31         3.51
      1      0.912      2.14         3.00   14.5% ↓
  done · converged · 1 iters · reduced χ² 3.00 · Rwp 2.14 % · 1.02 s


View(html=<113,234 characters>, height=376)

In [18]:
project.display.pattern('polaris')

View(html=<617,172 characters>, height=480)

### Displacements

In [19]:
for site in structure.atom_sites:
    site.adp_iso.free = True

In [20]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_cecoal-polaris, 3692 pts fitted / 3693 loaded, 30 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        2.14         3.01
      1      0.914      2.10         2.89   3.7% ↓
  done · converged · 1 iters · reduced χ² 2.89 · Rwp 2.10 % · 1.02 s


View(html=<119,045 characters>, height=376)

In [21]:
project.display.pattern('polaris')

View(html=<617,068 characters>, height=480)

### Absorption

Absorption weakens the intensities with time of flight, as the displacements do with angle, so
it comes after them, on its own.

In [22]:
experiment.absorption.abscor1.free = True

In [23]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_cecoal-polaris, 3692 pts fitted / 3693 loaded, 31 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        2.10         2.90
      1      0.803      2.08         2.86   1.3% ↓
      2      0.972      2.08         2.86   
  done · converged · 2 iters · reduced χ² 2.86 · Rwp 2.08 % · 1.08 s


View(html=<120,217 characters>, height=376)

In [24]:
project.display.pattern('polaris')

View(html=<617,294 characters>, height=480)

### Al3 occupancy

Last, Al3's occupancy, which moves with its displacement.

In [25]:
structure.atom_sites['Al3'].occupancy.free = True

In [26]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_cecoal-polaris, 3692 pts fitted / 3693 loaded, 32 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        2.08         2.86
      1      0.925      2.08         2.85   0.2% ↓
  done · converged · 1 iters · reduced χ² 2.85 · Rwp 2.08 % · 1.04 s


View(html=<121,539 characters>, height=376)

In [27]:
project.display.pattern('polaris')

View(html=<617,463 characters>, height=480)

In [28]:
project.display.fit.correlations()

,1,2,3,4,5
1 cecoal.cell.length_b,,,,,
2 cecoal.cell.length_c,0.94,,,,
3 polaris.instrument.calib_d_to_tof_quadratic,-0.96,-0.95,,,
4 polaris.background.1.coef,0.00,0.00,-0.00,,
5 polaris.background.2.coef,0.00,0.00,-0.00,0.97,


In [29]:
project.display.structure('cecoal')

View(html=<230,276 characters>, height=660)

## Save the project

In [30]:
project.save_as('pd-neut-tof_cecoal-polaris')
project.report.save_html('pd-neut-tof_cecoal-polaris.html')

Saving project to 'pd-neut-tof_cecoal-polaris'
Saving report to 'pd-neut-tof_cecoal-polaris.html'
